In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
from datetime import date, datetime, time, timedelta
from plotnine import *

from ycn.backends.duckdb_backend import DuckDBSource

warnings.filterwarnings("ignore")

In [ ]:
# https://nbviewer.org/github/mayabenowitz/Hedgecraft/blob/master/notebooks/Hedgecraft.ipynb

In [ ]:
DUCKDB_PATH = r"D:\data\duckdb\equity_eod_data.duckdb"

db = DuckDBSource(DUCKDB_PATH, read_only=True)

df2 = db.run_query(
    """
    SELECT
        CAST("Index" AS DATE) AS Date,
        Stock AS Name,
        Open,
        High,
        Low,
        Close,
        Volume,
        EqIndex
    FROM equity_eod
    WHERE EqIndex = 'FTSE100'
    ORDER BY Date, Name
    """
)

df = db.run_query(
    """
    SELECT
        CAST("Index" AS DATE) AS Date,
        Stock AS Name,
        Open,
        High,
        Low,
        Close,
        Volume
    FROM equity_eod
    WHERE EqIndex = 'DAX30'
    ORDER BY Date, Name
    """
)

In [ ]:
df = df.with_columns(pl.col("Date").cast(pl.Date))
df2 = df2.with_columns(pl.col("Date").cast(pl.Date))

In [ ]:
df.sample(10) # no need to normalize for distance correlation (so if using that no need to use returns?)

In [ ]:
PRICE_COLS = ["Open", "High", "Low", "Close"]


def to_daily_returns(df: pl.DataFrame) -> pl.DataFrame:
    """Replace OHLC price columns with per-stock daily simple returns."""
    return (
        df.sort("Name", "Date")
        .with_columns(
            pl.col(col).pct_change().over("Name").alias(col) for col in PRICE_COLS
        )
        .drop_nulls(subset=PRICE_COLS)
    )


df = to_daily_returns(df)
df2 = to_daily_returns(df2)

In [ ]:
# imports the dcor module to calculate distance correlation
import dcor
from tqdm.auto import tqdm


def df_distance_correlation(df_train: pl.DataFrame, stocks: list[str]) -> pl.DataFrame:
    """Compute the distance-correlation matrix for wide-format stock columns."""
    dcor_values = {stock: {} for stock in stocks}
    n_pairs = sum(len(stocks[k:]) for k in range(len(stocks)))
    pbar = tqdm(total=n_pairs, desc="dcor")
    k = 0
    for i in stocks:
        v_i = df_train.get_column(i).to_numpy()
        for j in stocks[k:]:
            pbar.set_description(f"dcor: {i} / {j}")
            v_j = df_train.get_column(j).to_numpy()

            vi_vj = np.column_stack((v_i, v_j))
            vi_vj = vi_vj[~np.isnan(vi_vj).any(axis=1)]

            dcor_val = dcor.distance_correlation(vi_vj[:, 0], vi_vj[:, 1])
            dcor_values[i][j] = dcor_val
            dcor_values[j][i] = dcor_val
            pbar.update(1)
        k += 1
    pbar.close()

    return pl.DataFrame(
        {row: [dcor_values[row][col] for col in stocks] for row in stocks}
    )

In [ ]:
# dff will be the df2
dff = df2.clone()

In [ ]:
(dff["Date"].min(), dff["Date"].max())

In [ ]:
# creates a DataFrame for each time-series (see In [11])
cutoff_date = date(2012, 1, 1)
df_train = (
    dff.filter(pl.col("Date") < cutoff_date)
    .drop_nulls()
    .with_columns((pl.col("Close") - pl.col("Open")).alias("IntraDelta"))
)

In [ ]:
def pivot_to_wide_format(
    df: pl.DataFrame,
    date_col: str = "Date",
    name_col: str = "Name",
    value_col: str = "Close",
) -> pl.DataFrame:
    """Pivot a long dataframe to wide format with dates as rows."""
    df_pivoted = df.pivot(on=name_col, index=date_col, values=value_col).sort(date_col)
    stock_cols = sorted(col for col in df_pivoted.columns if col != date_col)
    return df_pivoted.select([date_col, *stock_cols])


df_train_close = pivot_to_wide_format(df_train, value_col="Close")
df_train_open = pivot_to_wide_format(df_train, value_col="Open")
df_train_high = pivot_to_wide_format(df_train, value_col="High")
df_train_low = pivot_to_wide_format(df_train, value_col="Low")
df_train_intradelta = pivot_to_wide_format(df_train, value_col="IntraDelta")


In [ ]:
stock_cols = [col for col in df_train_close.columns if col != "Date"]

# df_train_high_dcor = df_distance_correlation(df_train_high, stocks=stock_cols)
# df_train_low_dcor = df_distance_correlation(df_train_low, stocks=stock_cols)
# df_train_open_dcor = df_distance_correlation(df_train_open, stocks=stock_cols)
df_train_close_dcor = df_distance_correlation(df_train_close, stocks=stock_cols)
df_train_intradelta_dcor = df_distance_correlation(
    df_train_intradelta, stocks=stock_cols
)

In [ ]:
#imports the NetworkX module
import networkx as nx

# takes in a pre-processed dataframe and returns a time-series correlation
# network with pairwise distance correlation values as the edges
def build_corr_nx(df_train: pl.DataFrame, independent_threshold: float = 0.33):
    # converts the distance correlation dataframe to a numpy matrix with dtype float
    cor_matrix = df_train.to_numpy().astype(float)
    # Since dcor ranges between 0 and 1, (0 corresponding to independence and 1
    # corresponding to dependence), 1 - cor_matrix results in values closer to 0
    # indicating a higher degree of dependence where values close to 1 indicate a lower degree of 
    # dependence. This will result in a network with nodes in close proximity reflecting the similarity
    # of their respective time-series and vice versa.
    sim_matrix = 1 - cor_matrix
    # transforms the similarity matrix into a graph
    G = nx.from_numpy_array(sim_matrix)
    # extracts the stock names from the dataframe columns
    stock_names = np.array(df_train.columns)
    # relabels the nodes of the network with the stock names
    G = nx.relabel_nodes(G, lambda x: stock_names[x])
    # assigns the edges of the network weights (i.e., the dcor values)
    G.edges(data=True)
    # copies G
    ## we need this to delete edges or othwerwise modify G
    H = G.copy()
    # iterates over the edges of H (the u-v pairs) and the weights (wt)
    for (u, v, wt) in G.edges.data('weight'):
        # selects edges with dcor values less than or equal to 0.33
        if wt >= 1 - independent_threshold:  
            # removes the edges 
            H.remove_edge(u, v)
        # selects self-edges
        if u == v:
            # removes the self-edges
            H.remove_edge(u, v)
    # returns the final stock correlation network            
    return H

In [ ]:
# df_train_high_dcor

In [ ]:
H_close = build_corr_nx(df_train_close_dcor)
# H_open = build_corr_nx(df_train_open_dcor)
H_close_diff = build_corr_nx(df_train_intradelta_dcor)
# H_high = build_corr_nx(df_train_high_dcor)
# H_low = build_corr_nx(df_train_low_dcor)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
# def plot_corr_nx_w_title(H, df_train, type = "Close"):
#     title_str = f"Distance Correlation Network of the Daily {type} ({df_train['Date'].min().strftime('%Y-%m-%d')}-{df_train['Date'].max().strftime('%Y-%m-%d')})"
#     plt_corr_nx(H, title=title_str)

# function to display the network from the distance correlation matrix
def plot_corr_nx(H, title):

    # creates a set of tuples: the edges of G and their corresponding weights
    edges, weights = zip(*nx.get_edge_attributes(H, "weight").items())

    pos = nx.kamada_kawai_layout(H)

    with sns.axes_style('whitegrid'):
        # figure size and style
        plt.figure(figsize=(12, 9))
        plt.title(title, size=16)

        # computes the degree (number of connections) of each node
        deg = H.degree

        # list of node names
        nodelist = []
        # list of node sizes
        node_sizes = []

        # iterates over deg and appends the node names and degrees
        for n, d in deg:
            nodelist.append(n)
            node_sizes.append(d)

        # draw nodes
        nx.draw_networkx_nodes(
            H,
            pos,
            node_color="#DA70D6",
            nodelist=nodelist,
            node_size=np.power(node_sizes, 2.33),
            alpha=0.8,
            # font_weight="bold",
        )

        # node label styles
        nx.draw_networkx_labels(H, pos, font_size=13, font_family="sans-serif", font_weight='bold')

        # color map
        cmap = sns.cubehelix_palette(3, as_cmap=True, reverse=True)

        # draw edges
        nx.draw_networkx_edges(
            H,
            pos,
            edgelist=edges,
            style="solid",
            edge_color=weights,
            edge_cmap=cmap,
            edge_vmin=min(weights),
            edge_vmax=max(weights),
        )

        # builds a colorbar
        sm = plt.cm.ScalarMappable(
            cmap=cmap, 
            norm=plt.Normalize(vmin=min(weights), 
            vmax=max(weights))
        )
        sm._A = []
        plt.colorbar(sm)

        # displays network without axes
        plt.axis("off")

#silence warnings   
import warnings
warnings.filterwarnings("ignore")


In [ ]:

def plot_corr_nx(H, title="Distance Correlation Network"):
    """
    Plot a networkx graph with edge colors based on weights.
    Fixed version with proper colorbar axes handling.
    
    Parameters:
    -----------
    H : networkx.Graph
        Graph to plot
    title : str, default "Distance Correlation Network"
        Title for the plot
    """
    # Create figure and axes explicitly
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Get edge weights
    weights = [H[u][v]['weight'] for u, v in H.edges()]
    
    # Set up colormap
    cmap = plt.cm.viridis
    
    # Draw the graph
    # This draws the network with the Kamada-Kawai path-length cost-function.
    # Nodes are positioned by treating the network as a physical ball-and-spring system. The locations
    # of the nodes are such that the total energy of the system is minimized.
    pos = nx.kamada_kawai_layout(H)
    # pos = nx.spring_layout(H, k=1, iterations=50)
    edges = nx.draw_networkx_edges(H, pos, edge_color=weights, edge_cmap=cmap, 
                                   width=2, alpha=0.6, ax=ax)
    nx.draw_networkx_nodes(H, pos, node_color='lightblue', node_size=500, ax=ax)
    nx.draw_networkx_labels(H, pos, font_size=8, ax=ax)
    
    # Create ScalarMappable for colorbar
    sm = plt.cm.ScalarMappable(
        cmap=cmap, 
        norm=plt.Normalize(vmin=min(weights), vmax=max(weights))
    )
    sm.set_array([])  # Fixed: use set_array instead of _A
    
    # Add colorbar with explicit axes reference
    plt.colorbar(sm, ax=ax)
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_corr_nx(H_close, 'Close')

In [ ]:
import base64
import re

from IPython.display import HTML, display
from pyvis.network import Network


def _weight_to_color(norm_weight: float) -> str:
    """Map a normalized weight in [0, 1] to a viridis-like hex color."""
    if norm_weight < 0.5:
        r = 0
        g = int(norm_weight * 2 * 255)
        b = 255
    else:
        r = int((norm_weight - 0.5) * 2 * 255)
        g = 255
        b = int(255 - (norm_weight - 0.5) * 2 * 255)
    return f"#{r:02x}{g:02x}{b:02x}"


def _clean_pyvis_html(html: str) -> str:
    """Remove pyvis template quirks before embedding in the notebook."""
    # The bundled template renders the heading twice.
    html = re.sub(r"<center>\s*<h1>.*?</h1>\s*</center>\s*", "", html, count=1, flags=re.S)
    return html


def plot_corr_pyvis(
    H,
    title: str = "Distance Correlation Network",
    height: str = "700px",
    width: str = "100%",
):
    """Plot a NetworkX graph with pyvis in an isolated iframe."""
    if H.number_of_edges() == 0:
        print("Graph has no edges to plot.")
        return

    weights = [data["weight"] for _, _, data in H.edges(data=True)]
    min_weight = min(weights)
    max_weight = max(weights)

    def normalize_weight(weight: float) -> float:
        if max_weight == min_weight:
            return 0.5
        return (weight - min_weight) / (max_weight - min_weight)

    n_nodes = H.number_of_nodes()
    spring_length = max(220, min(450, 100 + n_nodes * 3))
    repulsion = -25000 - n_nodes * 150

    net = Network(
        height=height,
        width=width,
        notebook=True,
        cdn_resources="remote",
        heading=title,
        bgcolor="#ffffff",
    )
    net.barnes_hut(
        gravity=repulsion,
        central_gravity=0.03,
        spring_length=spring_length,
        spring_strength=0.04,
        damping=0.25,
        overlap=0.8,
    )

    for node in H.nodes():
        net.add_node(
            node,
            label=str(node),
            size=16,
            color="#ADD8E6",
            title=str(node),
            font={"size": 14, "face": "arial bold"},
        )

    for u, v, data in H.edges(data=True):
        weight = data["weight"]
        norm_weight = normalize_weight(weight)
        net.add_edge(
            u,
            v,
            width=0.4 + norm_weight * 0.6,
            color=_weight_to_color(norm_weight),
            title=f"weight: {weight:.3f}",
        )

    net.set_edge_smooth("continuous")
    net.toggle_physics(True)

    html = _clean_pyvis_html(net.generate_html(notebook=True))
    encoded = base64.b64encode(html.encode("utf-8")).decode("ascii")
    iframe = (
        f'<iframe src="data:text/html;base64,{encoded}" '
        f'width="{width}" height="{height}" frameborder="0" '
        'style="border:1px solid #ddd; background:#fff;"></iframe>'
    )
    display(HTML(iframe))


In [ ]:
plot_corr_pyvis(H_close, title="Distance Correlation Network (Close)")

In [ ]:
# function to visualize the degree distribution
def hist_plot(network, title, bins, xticks):
    
    # extracts the degrees of each vertex and stores them as a list
    deg_list = list(dict(network.degree).values())
    # sets local style
    with plt.style.context('fivethirtyeight'):
        # initializes a figure
        plt.figure(figsize=(9,6))
        # plots a pretty degree histogram with a kernel density estimator
        sns.distplot(
            deg_list,  
            kde=True,
            bins = bins,
            color='darksalmon',
            hist_kws={'alpha': 0.7}
        );
        # turns the grid off
        plt.grid(True)
        # controls the number and spacing of xticks and yticks
        plt.xticks(xticks, size=11)
        plt.yticks(size=11)
        # removes the figure spines
        #  sns.despine(left=True, right=True, bottom=True, top=True)
        # labels the y and x axis
        #  plt.ylabel("Probability", size=15)
        plt.xlabel("Number of Connections", size=15)
        # sets the title
        plt.title(title, size=20);
        # draws a vertical line where the mean is
        plt.axvline(sum(deg_list)/len(deg_list), 
                    color='darkorchid', 
                    linewidth=3, 
                    linestyle='--', 
                    label='Mean = {:2.0f}'.format(sum(deg_list)/len(deg_list))
        )

        # turns the legend on
        plt.legend(loc=0, fontsize=12)

In [ ]:
# plots the degree histogram of the closing prices network
hist_plot(
    H_close, 
    'Degree Histogram of the Closing Prices Network', 
    bins=15, 
    xticks=range(13, 30, 2)
)


In [ ]:
H_close.degree